# Notebook 03 — Annotation

## Purpose

Document the manual annotation file, the labels actually stored in it, and how those 1,199 records relate to the 5,292-comment corpus.

This notebook does not change labels and does not train a classifier.

## Inputs

- `Annotation Dataset/NIRA_Annotation_Master_1199_Modeling_final.xlsx`
- `outputs/01_data_preparation/20261018_content_review/annotation_linked.csv`
- `outputs/01_data_preparation/20261018_content_review/text_mismatch_register.csv`
- `outputs/01_data_preparation/20261018_content_review/raw_corpus_working.csv`

## Boundary

No written codebook was found in the project folder. The categories below are the values stored in the workbook. This notebook does not invent definitions for them.


## 0. Load the annotation workbook and the linkage outputs

Use the annotation file supplied by the researcher and the keys already matched in notebook 01.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.project_config import (
    ANNOTATION_COLUMNS,
    ANNOTATION_XLSX,
    CLASSIFICATION_TARGETS,
    EXPECTED_ANNOTATION_N,
    EXPECTED_RAW_N,
)

RUN_IN = ROOT / "outputs" / "01_data_preparation" / "20261018_content_review"
RUN_ID = "20261018_annotation"
OUT = ROOT / "outputs" / "03_annotation" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

source = pd.read_excel(ANNOTATION_XLSX, dtype=str)
linked = pd.read_csv(RUN_IN / "annotation_linked.csv", dtype=str)
raw = pd.read_csv(RUN_IN / "raw_corpus_working.csv")
mismatch = pd.read_csv(RUN_IN / "text_mismatch_register.csv", dtype=str)

if len(source) != EXPECTED_ANNOTATION_N or len(linked) != EXPECTED_ANNOTATION_N:
    raise ValueError("Annotation row count is not 1,199.")
if len(raw) != EXPECTED_RAW_N:
    raise ValueError("Raw working corpus is not 5,292 rows.")
missing = [col for col in ANNOTATION_COLUMNS if col not in source.columns]
if missing:
    raise ValueError(f"Annotation file is missing columns: {missing}")
if not set(linked["record_key"]).issubset(set(raw["record_key"].astype(str))):
    raise ValueError("Annotated keys are not all inside the raw corpus.")

print("Annotation source rows:", len(source))
print("Linked rows:", len(linked))
print("Classification targets:", CLASSIFICATION_TARGETS)
print("Output:", OUT)


Annotation source rows: 1199
Linked rows: 1199
Classification targets: ['policy_relevance', 'feedback_type', 'topic', 'sentiment']
Output: D:\NIRA_CASE_STUDY\outputs\03_annotation\20261018_annotation


### Interpretation

The load confirms 1,199 annotation rows and 5,292 raw comments. The linked keys are a subset of the raw keys, so every annotated comment can be traced to one source comment.


## 1. Fields in the workbook

Four fields are the supervised classification tasks. `actor` is stored in the file and is not one of those four tasks. `annotation_status` records that the row was annotated. It is not a second coder's judgement.


In [2]:
empty_counts = {
    col: int((source[col].fillna("").astype(str).str.strip() == "").sum())
    for col in ANNOTATION_COLUMNS
}
if any(empty_counts[col] for col in CLASSIFICATION_TARGETS + ["annotation_id", "comment", "actor", "annotation_status"]):
    raise ValueError(f"Unexpected empty annotation fields: {empty_counts}")

schema_rows = []
roles = {
    "annotation_id": "Identifier for the annotated row. Unique in this file.",
    "comment_id": "Identifier within a source post. Not unique across posts.",
    "comment": "Comment text stored in the annotation file. Compared with the raw text below.",
    "platform": "Platform value stored with the annotation.",
    "post_url": "Source-post URL stored with the annotation.",
    "collection_date": "Collection timestamp stored in the annotation file.",
    "collection_date_original": "Earlier string form of the collection timestamp. Not a publication date.",
    "policy_relevance": "Supervised target.",
    "feedback_type": "Supervised target.",
    "topic": "Supervised target. This is the manual topic label, not an unsupervised topic.",
    "sentiment": "Supervised target.",
    "actor": "Stored label. Not one of the four supervised tasks.",
    "annotation_status": "Status stored on the row. Every row in this file says annotated.",
}
for col in ANNOTATION_COLUMNS:
    schema_rows.append({
        "field": col,
        "role": roles[col],
        "n_nonempty": int((source[col].fillna("").astype(str).str.strip() != "").sum()),
        "n_distinct_values": int(source[col].nunique(dropna=False)),
    })
schema = pd.DataFrame(schema_rows)
print(schema.to_string(index=False))
print("annotation_status values:", sorted(source["annotation_status"].unique()))
print("No second-coder column and no agreement score are present in this file.")


                   field                                                                          role  n_nonempty  n_distinct_values
           annotation_id                        Identifier for the annotated row. Unique in this file.        1199               1199
              comment_id                     Identifier within a source post. Not unique across posts.        1199                519
                 comment Comment text stored in the annotation file. Compared with the raw text below.        1199               1199
                platform                                    Platform value stored with the annotation.        1199                  1
                post_url                                   Source-post URL stored with the annotation.        1199                 20
         collection_date                           Collection timestamp stored in the annotation file.        1199                 19
collection_date_original      Earlier string form of the colle

### Interpretation

Four fields are the supervised tasks: `policy_relevance`, `feedback_type`, `topic`, and `sentiment`. `actor` is present but is not one of those tasks. No task field is empty, and the status column contains only `annotated`. The researcher later confirmed that a separate codebook is not part of this study.


## 2. Observed labels

Shares use 1,199 annotated comments as the denominator. A written definition for each category was not found in the project folder.


In [3]:
frames = []
for field in CLASSIFICATION_TARGETS + ["actor"]:
    counts = source[field].value_counts().rename_axis("label").reset_index(name="n_comments")
    counts.insert(0, "field", field)
    counts["share_of_annotated"] = counts["n_comments"] / len(source)
    counts["minority_share"] = counts["n_comments"] / counts["n_comments"].max()
    frames.append(counts)
labels = pd.concat(frames, ignore_index=True)
print(labels.to_string(index=False))

imbalance = []
for field, part in labels.groupby("field"):
    imbalance.append({
        "field": field,
        "n_classes": int(part["label"].nunique()),
        "largest_class": str(part.iloc[0]["label"]),
        "largest_n": int(part.iloc[0]["n_comments"]),
        "smallest_class": str(part.iloc[-1]["label"]),
        "smallest_n": int(part.iloc[-1]["n_comments"]),
        "largest_share_of_annotated": float(part.iloc[0]["share_of_annotated"]),
    })
imbalance_df = pd.DataFrame(imbalance)
print(imbalance_df.to_string(index=False))


           field                         label  n_comments  share_of_annotated  minority_share
policy_relevance                      relevant         821            0.684737        1.000000
policy_relevance                  not_relevant         237            0.197665        0.288672
policy_relevance                     uncertain         141            0.117598        0.171742
   feedback_type                     complaint         317            0.264387        1.000000
   feedback_type                         other         232            0.193495        0.731861
   feedback_type                       support         172            0.143453        0.542587
   feedback_type                      question         162            0.135113        0.511041
   feedback_type                       request         153            0.127606        0.482650
   feedback_type                     criticism          98            0.081735        0.309148
   feedback_type                    experience    

### Interpretation

The label inventory uses 1,199 as its denominator. `relevant`, `complaint`, topic `other`, and `negative` are the largest classes. The smallest classes have 17 and 21 comments. That imbalance is why model selection uses macro-F1. One later correction, ANN_0031, changes sentiment from `positive` to `neutral`. It is stored in `annotation_corrections.csv` and was not written back into the original workbook.


## 3. Relationship to the raw corpus

The annotated records are a matched subset of the raw corpus. They are not a separate population.


In [4]:
match_counts = linked["text_match_status"].value_counts().rename_axis("text_match_status").reset_index(name="n_comments")
match_counts["share_of_annotated"] = match_counts["n_comments"] / len(linked)
note_counts = (
    mismatch["text_difference_note"].value_counts().rename_axis("text_difference_note").reset_index(name="n_comments")
)
print(match_counts.to_string(index=False))
print(note_counts.to_string(index=False))

coverage = (
    raw.groupby("post_id", as_index=False)
    .agg(n_comments=("record_key", "size"), n_annotated=("is_annotated", "sum"))
)
coverage["annotated_share_of_post"] = coverage["n_annotated"] / coverage["n_comments"]
if int(coverage["n_annotated"].sum()) != len(linked):
    raise ValueError("Annotated post totals do not equal 1,199.")

relation = {
    "annotated_comments": int(len(linked)),
    "raw_comments": int(len(raw)),
    "annotated_share_of_raw": len(linked) / len(raw),
    "unmatched_annotation_keys": 0,
    "exact_text_matches": int((linked["text_match_status"] == "exact").sum()),
    "newline_or_whitespace_differences": int((linked["text_match_status"] == "newline_or_whitespace").sum()),
    "content_differences": int((linked["text_match_status"] == "content_difference").sum()),
    "annotated_share_by_post_min": float(coverage["annotated_share_of_post"].min()),
    "annotated_share_by_post_max": float(coverage["annotated_share_of_post"].max()),
    "codebook_file_present": False,
    "inter_annotator_agreement_available": False,
    "decision": (
        "All 1,199 annotation keys match a raw comment. "
        "Twenty-five text differences are retained in the mismatch register and were not overwritten. "
        "One of those differences is a phone number redacted in the annotation file. "
        "The annotated share of each post is about 22 to 23 percent. "
        "No sampling protocol is stored in the project files."
    ),
}
print(relation["decision"])


    text_match_status  n_comments  share_of_annotated
                exact        1174            0.979149
   content_difference          13            0.010842
newline_or_whitespace          12            0.010008
                                          text_difference_note  n_comments
               difference is limited to line breaks or spacing          12
          one version contains the other plus extra characters           6
                              small character-level difference           4
          annotation contains the literal Excel marker _x000D_           2
annotation redacts material that is present in the raw comment           1
All 1,199 annotation keys match a raw comment. Twenty-five text differences are retained in the mismatch register and were not overwritten. One of those differences is a phone number redacted in the annotation file. The annotated share of each post is about 22 to 23 percent. No sampling protocol is stored in the project files.


### Interpretation

Text agreement is 1,174 exact matches, 12 spacing-only differences, and 13 character-level differences. The annotated share of each post stays between about 21.9 and 23.3 percent. The mismatch rows are retained rather than overwritten.


## 4. Save the annotation record

Later supervised notebooks should train on these stored labels. They should not merge rare classes unless a separate, documented decision says so.


In [5]:
def save_df(frame: pd.DataFrame, name: str) -> None:
    path = OUT / name
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(path)
    print("OK", name, len(frame))

save_df(schema, "annotation_schema.csv")
save_df(labels, "annotation_label_inventory.csv")
save_df(imbalance_df, "annotation_class_balance.csv")
save_df(match_counts, "annotation_text_match_counts.csv")
save_df(note_counts, "annotation_text_difference_notes.csv")
save_df(coverage, "annotation_coverage_by_post.csv")
(OUT / "annotation_relation.json").write_text(json.dumps(relation, indent=2), encoding="utf-8")
print("Relation file written.")
print("Annotated share of raw:", round(relation["annotated_share_of_raw"], 4))
print("Content differences retained:", relation["content_differences"])


OK annotation_schema.csv 13
OK annotation_label_inventory.csv 28
OK annotation_class_balance.csv 5
OK annotation_text_match_counts.csv 3
OK annotation_text_difference_notes.csv 5
OK annotation_coverage_by_post.csv 19
Relation file written.
Annotated share of raw: 0.2266
Content differences retained: 13


### Interpretation

The saved tables are the annotation record used by the modeling notebooks. They document the observed labels and the text differences. They do not add label definitions that were not in the files.


## 5. Consequences for the next notebooks

- Supervised training uses the 1,199 stored labels. The four tasks stay separate.
- `other` is the largest manual topic, `complaint` is the largest feedback type, and `negative` is the largest sentiment. Class sizes are unequal, so macro-F1 is the main comparison metric.
- The manual topic field is not the target of the unsupervised topic model. That model is fitted to all 5,292 comments.
- The 25 text differences stay documented. Training text should keep a stable choice, recorded in the modelling notebook, rather than silently preferring one version.
- No agreement statistic can be reported because the file contains one annotation status and no second coder.
